# QuickDraw AI training in Google Colab

Train the project CNN on a Colab GPU, then download `quickdraw.pt` to use with the local webcam, canvas, or API. In Colab choose **Runtime → Change runtime type → T4 GPU** (or another GPU) before running.

First create a source ZIP on your computer by running `scripts/create_colab_bundle.ps1` in PowerShell. Then upload that ZIP in the next cell. The notebook downloads the 10,000-example-per-class training data directly in Colab.

In [ ]:
from google.colab import files
from pathlib import Path
from zipfile import ZipFile
import shutil

uploaded = files.upload()  # Select quickdraw-colab-source.zip
zip_name = next(name for name in uploaded if name.lower().endswith('.zip'))
project_dir = Path('/content/quickdraw-ai')
if project_dir.exists():
    shutil.rmtree(project_dir)
project_dir.mkdir(parents=True)
with ZipFile(zip_name) as archive:
    archive.extractall(project_dir)
print('Project source extracted to', project_dir)

## Install training dependencies

Colab includes PyTorch. This installs the project package and only the additional libraries needed for training, leaving Colab's CUDA-enabled PyTorch intact.

In [ ]:
%cd /content/quickdraw-ai
!pip -q install -e . --no-deps
!pip -q install matplotlib scikit-learn
import torch
print('PyTorch:', torch.__version__)
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'not enabled')
assert torch.cuda.is_available(), 'Enable a GPU in Runtime > Change runtime type, then rerun this cell.'

## Download the dataset

This downloads 10,000 drawings for each of the 20 configured classes. To change the sample count, update `--count`.

In [ ]:
!python scripts/download_dataset.py --count 10000

## Train

The best validation checkpoint is written to `/content/quickdraw.pt`. Adjust `EPOCHS`, `BATCH_SIZE`, or `SAMPLES_PER_CLASS` to control the run.

In [ ]:
EPOCHS = 10
BATCH_SIZE = 256
SAMPLES_PER_CLASS = 10000
!python scripts/train.py --epochs $EPOCHS --batch-size $BATCH_SIZE --max-per-class $SAMPLES_PER_CLASS --output /content/quickdraw.pt --plots /content/training_curves.png

In [ ]:
from IPython.display import display, Image
display(Image('/content/training_curves.png'))

## Download the trained model

Run the cell below and save the downloaded `quickdraw.pt` into your local project's `models/` folder. The checkpoint includes model weights, class order, and validation metrics; no separate labels file is needed.

In [ ]:
files.download('/content/quickdraw.pt')

## Use it locally

Copy `quickdraw.pt` into `<project>/models/quickdraw.pt`, then run one of:

```powershell
python -m app.canvas --model models/quickdraw.pt
python -m app.webcam --model models/quickdraw.pt
uvicorn api.main:app --reload
```

You can optionally export ONNX locally with `python scripts/export_onnx.py --checkpoint models/quickdraw.pt`.